In [1]:
import crested

2025-03-15 13:03:02.805881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742043783.294097 1063461 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742043783.601462 1063461 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742043784.562902 1063461 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742043784.562928 1063461 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742043784.562930 1063461 computation_placer.cc:177] computation placer alr

In [2]:
%matplotlib inline

In [3]:
import tensorflow as tf

# Check GPU availability
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# Check TensorFlow build info
print(tf.sysconfig.get_build_info())

# Check CUDA version
print(tf.test.is_built_with_cuda())

# Check cuDNN version
print(tf.test.is_built_with_gpu_support())

Num GPUs Available: 1
OrderedDict([('cpu_compiler', '/usr/lib/llvm-18/bin/clang'), ('cuda_compute_capabilities', ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90']), ('cuda_version', '12.5.1'), ('cudnn_version', '9'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False)])
True
True


In [5]:
import argparse

In [4]:
adata = crested.import_bigwigs(
    bigwigs_folder={input folder},
    regions_file={input.file},
    target_region_width={numeric input},  # optionally, use a different width than the consensus regions file (500bp) for the .X values calculation
    target="count",  # or "max", "count", "logcount" --> what we will be predicting
)
adata

2025-03-15T13:08:11.930673+0000 WARNING Chromsizes file not provided. Will not check if regions are within chromosomes
2025-03-15T13:08:15.453894+0000 INFO Extracting values from 22 bigWig files...


AnnData object with n_obs × n_vars = 22 × 234908
    obs: 'file_path'
    var: 'chr', 'start', 'end'

In [ ]:
import json

# Open and load the JSON file
with open('/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/crested/fold0.json', "r") as file:
    data = json.load(file)

In [ ]:
crested.pp.train_val_test_split(
    adata, strategy="chr", val_chroms=["chr8", "chr10"], test_chroms=["chr9", "chr18", "chrX"]
)

# Alternatively, We can split randomly on the regions
# crested.pp.train_val_test_split(
#     adata, strategy="region", val_size=0.1, test_size=0.1, random_state=42
# )

print(adata.var["split"].value_counts())
adata.var

In [ ]:
/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/crested/v3

In [1]:
import os

In [2]:
os.path.join('aaa', 'bb')

'aaa/bb'

In [5]:
os.path.join('aaa', 'bbb', 'preprocessed_data.h5ad')

'aaa/bbb/preprocessed_data.h5ad'

In [13]:
import anndata
import crested
import keras

# Set the genome
genome = crested.Genome(
    "/home/bt392/rds/rds-bg200-hphi-gottgens/references/10x/refdata-cellranger-arc-mm10-2020-A-2.0.0/fasta/genome.fa", 
    "/home/bt392/rds/rds-bg200-hphi-gottgens/users/bt392/10_Eomes_invitro_gut/results/atac/chrombpnet/mm10.chrom.sizes"
)
crested.register_genome(
    genome
)  # Register the genome so that it can be used by the package


# Load the data
adata = anndata.read_h5ad('../../../../results/atac/crested/preprocessed_data.h5ad')

datamodule = crested.tl.data.AnnDataModule(
    adata
)

evaluator = crested.tl.Crested(data=datamodule)

# load an existing model
evaluator.load_model(
    "../../../../results/atac/crested/v2/trial1/checkpoints/17.keras",
    compile=True,
)

model = keras.models.load_model(
    "../../../../results/atac/crested/v2/trial1/checkpoints/17.keras", compile=False
)

2025-03-15T13:54:28.426370+0000 INFO Genome genome registered.


In [14]:
evaluator.predict(
    adata, model_name="checkpoint"
)

/tmp/ipykernel_1063461/3084214004.py:1: DeprecationWarning: The `predict` method is deprecated and will be removed from this class in a future version. Use the standalone function `tl.predict()` instead.
  evaluator.predict(
I0000 00:00:1742046870.731473 1065801 service.cc:152] XLA service 0x14aa3000da40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742046870.740492 1065801 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2025-03-15 13:54:31.361349: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1742046872.365216 1065801 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-03-15 13:54:33.696468: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_452', 112 bytes spill 

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/home/bt392/miniconda3/envs/crested/lib/python3.11/as

In [11]:
predictions = crested.tl.predict(adata, model)
adata.layers["predictions"] = predictions.T

2025-03-15 13:54:19.281214: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 7.40GiB (rounded to 7945528320)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-03-15 13:54:19.281268: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1058] BFCAllocator dump for GPU_0_bfc
2025-03-15 13:54:19.281280: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (256): 	Total Chunks: 33, Chunks in use: 33. 8.2KiB allocated for chunks. 8.2KiB in use in bin. 440B client-requested in use in bin.
2025-03-15 13:54:19.281287: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1065] Bin (512): 	Total Chunks: 0, Chunks in use: 0. 0B allocated for chunks. 0B in use in bin. 0B client-requested in use in bin.
2025-03-15 13:54:19.281

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

 size 2048 next 35
2025-03-15 13:54:19.281524: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f0009700 of size 2048 next 36
2025-03-15 13:54:19.281528: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f0009f00 of size 2048 next 37
2025-03-15 13:54:19.281531: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f000a700 of size 2048 next 34
2025-03-15 13:54:19.281535: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f000af00 of size 256 next 39
2025-03-15 13:54:19.281538: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f000b000 of size 2048 next 41
2025-03-15 13:54:19.281542: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f000b800 of size 2048 next 42
2025-03-15 13:54:19.281545: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1114] InUse at 14a9f000c000 of size 2048 next 43
2025-03-15 13:54:19.281549: I external/local_xla/xl

In [ ]:
%matplotlib inline
# plot predictions vs ground truth for a random region in the test set defined by index
idx = 21
region = test_df.index[idx]
print(region)
crested.pl.bar.region_predictions(adata, region, title="Predictions vs Ground Truth")

In [ ]:
crested.pl.heatmap.correlations_self(
    adata,
    title="Self Correlation Heatmap",
    x_label_rotation=90,
    width=5,
    height=5,
    vmax=1,
    vmin=-0.15,
)

In [4]:
# evaluate the model on the test set
evaluator.test()

I0000 00:00:1741945096.215283  617243 service.cc:152] XLA service 0x14563c017550 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1741945096.215653  617243 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2025-03-14 09:38:16.684153: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1741945097.455524  617243 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-03-14 09:38:18.620713: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 112 bytes spill stores, 112 bytes spill loads

2025-03-14 09:38:18.655980: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 

95/96 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - concordance_correlation_coefficient: 0.8231 - cosine_similarity: 0.9498 - loss: -0.2015 - mean_absolute_error: 1.6211 - mean_squared_error: 12.3693 - pearson_correlation: 0.8809 - pearson_correlation_log: 0.7957 - zero_penalty_metric: 99.5548

2025-03-14 09:39:13.832953: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 4 bytes spill stores, 4 bytes spill loads

2025-03-14 09:39:14.011866: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 288 bytes spill stores, 288 bytes spill loads

2025-03-14 09:39:14.032658: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 168 bytes spill stores, 168 bytes spill loads

2025-03-14 09:39:14.038234: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_684', 8 bytes spill stores, 8 bytes spill loads

2025-03-14 09:39:17.327735: E external/local

96/96 ━━━━━━━━━━━━━━━━━━━━ 96s 511ms/step - concordance_correlation_coefficient: 0.8232 - cosine_similarity: 0.9497 - loss: -0.2006 - mean_absolute_error: 1.6208 - mean_squared_error: 12.3430 - pearson_correlation: 0.8804 - pearson_correlation_log: 0.7955 - zero_penalty_metric: 99.8593
2025-03-14T09:39:51.308967+0000 INFO Test concordance_correlation_coefficient: 0.8270
2025-03-14T09:39:51.309638+0000 INFO Test cosine_similarity: 0.9494
2025-03-14T09:39:51.309930+0000 INFO Test loss: -0.1591
2025-03-14T09:39:51.310215+0000 INFO Test mean_absolute_error: 1.6060
2025-03-14T09:39:51.310498+0000 INFO Test mean_squared_error: 11.0950
2025-03-14T09:39:51.310787+0000 INFO Test pearson_correlation: 0.8567
2025-03-14T09:39:51.311062+0000 INFO Test pearson_correlation_log: 0.7858
2025-03-14T09:39:51.311756+0000 INFO Test zero_penalty_metric: 114.3194


In [5]:
# add predictions for model checkpoint to the adata
evaluator.predict(
    adata, model_name="checkpoint"
)  # adds the predictions to the adata.layers["checkpoint_15"]

/tmp/ipykernel_616993/1914005144.py:2: DeprecationWarning: The `predict` method is deprecated and will be removed from this class in a future version. Use the standalone function `tl.predict()` instead.
  evaluator.predict(


917/918 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step

2025-03-14 09:41:38.716358: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_452', 4 bytes spill stores, 4 bytes spill loads

2025-03-14 09:41:38.766932: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_452', 168 bytes spill stores, 168 bytes spill loads

2025-03-14 09:41:38.926603: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_452', 288 bytes spill stores, 288 bytes spill loads

2025-03-14 09:41:38.948352: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_452', 8 bytes spill stores, 8 bytes spill loads

2025-03-14 09:41:41.421831: E external/local

918/918 ━━━━━━━━━━━━━━━━━━━━ 123s 133ms/step
2025-03-14T09:42:04.064789+0000 INFO Adding predictions to anndata.layers[checkpoint].


In [1]:
output = '/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/crested/v3'
runname = 'test'

In [7]:
import keras
import glob
import re
import os
# Get all .keras files in the specified directory
model_files = glob.glob(os.path.join(output, runname, "checkpoints/*.keras"))

# Extract numbers from filenames
def extract_number(filename):
    match = re.search(r"(\d+)\.keras", filename)
    return int(match.group(1)) if match else -1  # Return -1 if no number is found

# Find the latest model
latest_model_file = max(model_files, key=extract_number) if model_files else None

model = keras.models.load_model(
    os.path.join(output, runname, 'checkpoints', latest_model_file), compile=False
)

I0000 00:00:1742054698.956630 1078663 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79077 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:41:00.0, compute capability: 8.0


In [8]:
model

<Functional name=functional, built=True>

In [4]:
model_files

[]